# Importing Libraries

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Metrics Impacting Churn Rate - Statistical Analysis

In [2]:
df = pd.read_csv("C:/Users/aksha/Documents/Github Datasets/Netflix/netflix_large_dataset_cleaned.csv")
df = df[(df["subscription_start_date"] >= "2022-01-01") & (df["subscription_start_date"] <= "2023-12-31")]

# Create binary churn column
df['churn_binary'] = (df['churn_status'] == 'Cancelled').astype(int)

print(f"Overall churn rate: {df['churn_binary'].mean():.1%}")

Overall churn rate: 17.3%


In [3]:
# ─────────────────────────────────────────────
# CATEGORICAL VARIABLES → Chi-Square Test
# ─────────────────────────────────────────────
# Tests whether churn rate differs significantly across categories.
# If p < 0.05, the variable is statistically significant.

cat_vars = ['gender', 'region', 'subscription_plan', 'payment_method',
            'discount_applied', 'content_type', 'device_type', 'time_of_day',
            'liked', 'recommendation_source']

print("=== CATEGORICAL VARIABLES ===")
for col in cat_vars:
    group = df.groupby(col)['churn_binary'].mean().sort_values(ascending=False)
    ct = pd.crosstab(df[col], df['churn_binary'])  # Contingency table
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    print(f"\n{col} (p={p:.4f}):")
    print(group.to_string())

=== CATEGORICAL VARIABLES ===

gender (p=0.5463):
gender
Male      0.174239
Female    0.171921

region (p=0.2308):
region
Asia             0.176846
Europe           0.176577
North America    0.173664
Africa           0.167910
South America    0.164142

subscription_plan (p=0.0000):
subscription_plan
Basic       0.254209
Standard    0.149378
Premium     0.076208

payment_method (p=0.8189):
payment_method
Card             0.174428
Wallet           0.173291
Bank Transfer    0.171540

discount_applied (p=0.4446):
discount_applied
No     0.174032
Yes    0.170827

content_type (p=0.2764):
content_type
Movie     0.175357
Series    0.171196

device_type (p=0.8588):
device_type
Laptop    0.175189
Mobile    0.173803
TV        0.172150
Tablet    0.169988

time_of_day (p=0.6306):
time_of_day
Night        0.175057
Evening      0.174682
Afternoon    0.173718
Morning      0.168905

liked (p=0.2345):
liked
No     0.175245
Yes    0.170720

recommendation_source (p=0.8812):
recommendation_source
Search 

In [4]:
# ─────────────────────────────────────────────
# NUMERIC VARIABLES → Independent Samples T-Test
# ─────────────────────────────────────────────
# Compares the mean of each numeric variable between
# Active vs Cancelled users.
# If p < 0.05, the difference in means is statistically significant.

num_vars = ['age_group', 'monthly_fee', 'watch_time_minutes', 'session_count',
            'completion_percentage', 'rating', 'days_since_last_watch',
            'avg_weekly_watch_time', 'content_diversity_score']

print("\n=== NUMERIC VARIABLES ===")
for col in num_vars:
    active  = df[df['churn_binary'] == 0][col]
    churned = df[df['churn_binary'] == 1][col]
    t_stat, p = stats.ttest_ind(active, churned)
    print(f"{col}: Active mean={active.mean():.2f}, "
          f"Churned mean={churned.mean():.2f}, p={p:.4f}")


=== NUMERIC VARIABLES ===
age_group: Active mean=36.36, Churned mean=36.24, p=0.5617
monthly_fee: Active mean=13.30, Churned mean=11.49, p=0.0000
watch_time_minutes: Active mean=87.07, Churned mean=75.09, p=0.0000
session_count: Active mean=2.43, Churned mean=2.03, p=0.0000
completion_percentage: Active mean=92.15, Churned mean=91.07, p=0.0000
rating: Active mean=3.42, Churned mean=3.40, p=0.1293
days_since_last_watch: Active mean=29.90, Churned mean=29.95, p=0.8158
avg_weekly_watch_time: Active mean=262.41, Churned mean=226.82, p=0.0000
content_diversity_score: Active mean=0.60, Churned mean=0.60, p=0.1791


# Predicting Churn Rate for 2024 using a Random Forest

In [5]:
# Load data
df = pd.read_csv("C:/Users/aksha/Documents/Github Datasets/Netflix/netflix_large_dataset_cleaned.csv")
df = df[(df["subscription_start_date"] >= "2022-01-01") & (df["subscription_start_date"] <= "2023-12-31")]

# Convert churn_status to binary (1 = Cancelled, 0 = Active)
df['churn_binary'] = (df['churn_status'] == 'Cancelled').astype(int)

# Select features
features = ['age_group', 'gender', 'region', 'subscription_plan',
            'payment_method', 'discount_applied', 'content_type',
            'device_type', 'watch_time_minutes', 'session_count',
            'completion_percentage', 'time_of_day', 'rating',
            'liked', 'recommendation_source', 'days_since_last_watch',
            'avg_weekly_watch_time', 'content_diversity_score']

X_train = df[features].copy()
y_train = df['churn_binary']

# Encode categorical columns to numeric
le = LabelEncoder()
for col in X_train.select_dtypes(include='object').columns:
    X_train[col] = le.fit_transform(X_train[col].astype(str))

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Extract and display feature importances
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

avg_weekly_watch_time      0.146406
content_diversity_score    0.127594
days_since_last_watch      0.124826
watch_time_minutes         0.109524
age_group                  0.059433
recommendation_source      0.053398
region                     0.052631
completion_percentage      0.049502
time_of_day                0.047030
device_type                0.045545
rating                     0.041245
payment_method             0.035915
content_type               0.022183
gender                     0.021888
discount_applied           0.019372
subscription_plan          0.016831
liked                      0.013460
session_count              0.013218
dtype: float64


In [6]:
df2 = pd.read_csv("C:/Users/aksha/Documents/Github Datasets/Netflix/netflix_large_dataset_cleaned.csv")
df2 = df2[(df2["subscription_start_date"] >= "2024-01-01")]

# Convert churn_status to binary (1 = Cancelled, 0 = Active)
df2['churn_binary'] = (df2['churn_status'] == 'Cancelled').astype(int)

X_test = df2[features].copy()
y_test = df2['churn_binary']

# Encode categorical columns to numeric
le = LabelEncoder()
for col in X_test.select_dtypes(include='object').columns:
    X_test[col] = le.fit_transform(X_test[col].astype(str))

rf_preds = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

# Print results
print(f"Random Forest Accuracy: {rf_accuracy:.2%}")

Random Forest Accuracy: 83.17%
